### 1. Data Download & Cleaning & EDA

In [20]:
import pandas as pd
import numpy as np
from IPython.display import display
from pathlib import Path
from dataclasses import dataclass

#### Data download.

In [2]:
DATA_PATH = '../data/raw/ethusdt_1h.csv'
data = pd.read_csv(DATA_PATH)

#### Checking out my data -> a little overview.

In [3]:
data.shape

(46258, 10)

In [4]:
display(data.columns)

Index(['timestamp', 'open', 'high', 'low', 'close', 'volume',
       'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume',
       'taker_buy_quote_asset_volume'],
      dtype='str')

In [5]:
display(data.head(10))

,timestamp,open,high,low,close,volume,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume
0,2021-01-01T00:00:00Z,736.42,739.00,729.33,734.07,27932.69884,2.047990e+07,22671,15020.61663,1.101592e+07
1,2021-01-01T01:00:00Z,734.08,749.00,733.37,748.28,52336.18779,3.889962e+07,41712,27395.90270,2.036237e+07
2,2021-01-01T02:00:00Z,748.27,749.00,742.27,744.06,33019.50100,2.460673e+07,19566,17721.26696,1.320712e+07
3,2021-01-01T03:00:00Z,744.06,747.23,743.10,744.82,17604.80859,1.311909e+07,12230,9499.74920,7.079926e+06
4,2021-01-01T04:00:00Z,744.87,747.09,739.30,742.29,18794.15424,1.398019e+07,13626,10133.74211,7.539500e+06
5,2021-01-01T05:00:00Z,742.34,743.23,739.50,740.65,14948.26447,1.107935e+07,11032,7733.14162,5.731472e+06
6,2021-01-01T06:00:00Z,740.72,743.25,737.04,739.97,17106.99495,1.265847e+07,12041,7803.15217,5.774894e+06
7,2021-01-01T07:00:00Z,739.87,740.51,734.40,737.38,21624.68945,1.593471e+07,19976,10407.77312,7.670628e+06
8,2021-01-01T08:00:00Z,737.37,738.48,725.10,730.07,52992.04892,3.871314e+07,38433,23184.47715,1.693733e+07
9,2021-01-01T09:00:00Z,730.07,734.77,728.77,733.68,22836.46973,1.673203e+07,27347,12351.59569,9.050057e+06


In [6]:
display(data.tail(10))


,timestamp,open,high,low,close,volume,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume
46248,2026-04-12T14:00:00Z,2186.98,2193.53,2178.69,2180.86,16800.0145,3.674250e+07,139893,8708.7399,1.905615e+07
46249,2026-04-12T15:00:00Z,2180.88,2188.89,2175.00,2186.47,8622.2898,1.881263e+07,85325,4330.9882,9.452336e+06
46250,2026-04-12T16:00:00Z,2186.47,2190.43,2183.09,2187.09,4674.3927,1.022122e+07,53716,2211.3340,4.836474e+06
46251,2026-04-12T17:00:00Z,2187.09,2205.00,2187.09,2203.17,7545.5178,1.657255e+07,80626,5065.4047,1.112558e+07
46252,2026-04-12T18:00:00Z,2203.18,2207.23,2199.56,2200.60,5588.7032,1.231216e+07,59534,2846.4832,6.271431e+06
46253,2026-04-12T19:00:00Z,2200.60,2205.20,2197.40,2198.27,3607.8964,7.943680e+06,34095,1183.2717,2.605119e+06
46254,2026-04-12T20:00:00Z,2198.27,2218.13,2193.34,2213.53,8794.3235,1.940547e+07,91423,4870.5427,1.075138e+07
46255,2026-04-12T21:00:00Z,2213.54,2217.71,2205.19,2211.24,4865.7070,1.075219e+07,72652,2352.9282,5.199985e+06
46256,2026-04-12T22:00:00Z,2211.24,2211.26,2183.99,2199.37,16989.2668,3.727669e+07,284928,8753.0787,1.920838e+07
46257,2026-04-12T23:00:00Z,2199.36,2199.36,2186.35,2191.65,6791.1234,1.487883e+07,126484,2650.2606,5.805765e+06


#### Checking out empty/corrupted data

In [7]:
empty_cols = data.isna().sum()
empty_cols

timestamp                       0
open                            0
high                            0
low                             0
close                           0
volume                          0
quote_asset_volume              0
number_of_trades                0
taker_buy_base_asset_volume     0
taker_buy_quote_asset_volume    0
dtype: int64

In [8]:
duplicates = data.duplicated().sum()
duplicates

np.int64(0)

#### Timestamp checks

In [16]:
ts = pd.to_datetime(data["timestamp"], utc=True, errors="coerce")
ts_info = pd.DataFrame({
    "min timestamp": ts.min(),
    "max timestamp": ts.max(),
    "count": ts.count(),
    "unique": ts.nunique(),
    "missing": ts.isna().sum(),
    "duplicates": ts.duplicated().sum()
}, index=["timestamp"])

display(ts_info)


full_idx = pd.date_range(ts.min(), ts.max(), freq="h", tz="UTC")
missing_hours = full_idx.difference(pd.DatetimeIndex(ts.dropna()))
ts_time_gaps = pd.DataFrame({
    "missing_hours": missing_hours,
    "gap_length": missing_hours.to_series().diff().dt.total_seconds() / 3600
})
display(ts_time_gaps)

,min timestamp,max timestamp,count,unique,missing,duplicates
timestamp,2021-01-01 00:00:00+00:00,2026-04-12 23:00:00+00:00,46258,46258,0,0


,missing_hours,gap_length
2021-02-11 04:00:00+00:00,2021-02-11 04:00:00+00:00,NaN
2021-03-06 02:00:00+00:00,2021-03-06 02:00:00+00:00,550.0
2021-04-20 02:00:00+00:00,2021-04-20 02:00:00+00:00,1080.0
2021-04-20 03:00:00+00:00,2021-04-20 03:00:00+00:00,1.0
2021-04-25 05:00:00+00:00,2021-04-25 05:00:00+00:00,122.0
2021-04-25 06:00:00+00:00,2021-04-25 06:00:00+00:00,1.0
2021-04-25 07:00:00+00:00,2021-04-25 07:00:00+00:00,1.0
2021-08-13 02:00:00+00:00,2021-08-13 02:00:00+00:00,2635.0
2021-08-13 03:00:00+00:00,2021-08-13 03:00:00+00:00,1.0
2021-08-13 04:00:00+00:00,2021-08-13 04:00:00+00:00,1.0


In [ ]:
ts = pd.to_datetime(data["timestamp"], utc=True, errors="coerce")
ts_info = pd.DataFrame({
    "min timestamp": ts.min(),
    "max timestamp": ts.max(),
    "count": ts.count(),
    "unique": ts.nunique(),
    "missing": ts.isna().sum(),
    "duplicates": ts.duplicated().sum()
}, index=["timestamp"])

display(ts_info)


full_idx = pd.date_range(ts.min(), ts.max(), freq="h", tz="UTC")
missing_hours = full_idx.difference(pd.DatetimeIndex(ts.dropna()))
ts_time_gaps = pd.DataFrame({
    "missing_hours": missing_hours,
    "gap_length": missing_hours.to_series().diff().dt.total_seconds() / 3600
})
display(ts_time_gaps)

,min timestamp,max timestamp,count,unique,missing,duplicates
timestamp,2021-01-01 00:00:00+00:00,2026-04-12 23:00:00+00:00,46258,46258,0,0


,missing_hours,gap_length
2021-02-11 04:00:00+00:00,2021-02-11 04:00:00+00:00,NaN
2021-03-06 02:00:00+00:00,2021-03-06 02:00:00+00:00,550.0
2021-04-20 02:00:00+00:00,2021-04-20 02:00:00+00:00,1080.0
2021-04-20 03:00:00+00:00,2021-04-20 03:00:00+00:00,1.0
2021-04-25 05:00:00+00:00,2021-04-25 05:00:00+00:00,122.0
2021-04-25 06:00:00+00:00,2021-04-25 06:00:00+00:00,1.0
2021-04-25 07:00:00+00:00,2021-04-25 07:00:00+00:00,1.0
2021-08-13 02:00:00+00:00,2021-08-13 02:00:00+00:00,2635.0
2021-08-13 03:00:00+00:00,2021-08-13 03:00:00+00:00,1.0
2021-08-13 04:00:00+00:00,2021-08-13 04:00:00+00:00,1.0


#### Candles integrity check

In [29]:
num_df = data.copy()
for c in ["open", "high", "low", "close", "volume"]:
    num_df[c] = pd.to_numeric(num_df[c], errors="coerce")

invalid_high = int((num_df["high"] < num_df[["open", "close"]].max(axis=1)).sum())
invalid_low = int((num_df["low"] > num_df[["open", "close"]].min(axis=1)).sum())
negative_volume = int((num_df["volume"] < 0).sum())

candles_integrity = pd.DataFrame({
    "invalid_high": invalid_high,
    "invalid_low": invalid_low,
    "negative_volume": negative_volume
}, index=["candles_integrity"])
display(candles_integrity)

,invalid_high,invalid_low,negative_volume
candles_integrity,0,0,0


#### Features split

In [24]:
@dataclass
class DataConfig:
    raw_csv: Path = Path("data/raw/ethusdt_1h.csv")
    clean_csv: Path = Path("data/interim/eth_clean.csv")
    features_parquet: Path = Path("data/processed/features.parquet")
    train_parquet: Path = Path("data/processed/train.parquet")
    val_parquet: Path = Path("data/processed/val.parquet")
    test_parquet: Path = Path("data/processed/test.parquet")
    
@dataclass
class TargetConfig:
    horizon: int = 6
    threshold: float = 0.0075
    sell_label: int = 0
    hold_label: int = 1
    buy_label: int = 2


@dataclass
class SplitConfig:
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    n_splits_cv: int = 5

In [26]:
def prepare_datasets(
    data_config: DataConfig,
    target_config: TargetConfig,
    split_config: SplitConfig,
    corr_threshold: float = 0.95,
    apply_vif: bool = True,
    vif_threshold: float = 10.0,
) -> dict:
    raw_df = load_ohlcv_csv(data_config.raw_csv)
    clean_df, cleaning_stats = clean_ohlcv(raw_df)
    save_dataframe(clean_df, data_config.clean_csv)

    feat_df = create_features(clean_df)
    feat_df = make_multiclass_target(
        feat_df,
        horizon=target_config.horizon,
        threshold=target_config.threshold,
    )
    feat_df = feat_df.dropna().copy()

    target_series = feat_df["target"].astype(int)
    future_return = feat_df["future_return"].copy()
    features_only = feat_df.drop(columns=["target", "future_return"])

    # Keep only numeric features for MLP input.
    features_only = features_only.select_dtypes(include=["number"]).copy()
    features_only, dropped_constant = drop_constant_features(features_only)
    features_only, dropped_corr = drop_highly_correlated_features(features_only, threshold=corr_threshold)

    dropped_vif: list[str] = []
    if apply_vif:
        features_only, dropped_vif = reduce_vif_features(features_only, vif_threshold=vif_threshold)

    final_df = features_only.copy()
    final_df["future_return"] = future_return.loc[final_df.index]
    final_df["target"] = target_series.loc[final_df.index]
    final_df = final_df.dropna().copy()

    train_df, val_df, test_df = chronological_train_val_test_split(
        final_df,
        train_ratio=split_config.train_ratio,
        val_ratio=split_config.val_ratio,
        test_ratio=split_config.test_ratio,
    )

    save_dataframe(final_df, data_config.features_parquet)
    save_dataframe(train_df, data_config.train_parquet)
    save_dataframe(val_df, data_config.val_parquet)
    save_dataframe(test_df, data_config.test_parquet)

    reports_dir = data_config.features_parquet.parents[2] / "reports"
    ensure_dir(reports_dir)

    class_counts = final_df["target"].value_counts().sort_index().to_dict()
    metadata = {
        "cleaning_stats": cleaning_stats,
        "rows_final_dataset": len(final_df),
        "split_sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
        "class_counts": {str(k): int(v) for k, v in class_counts.items()},
        "num_features": int(len(features_only.columns)),
        "feature_columns": list(features_only.columns),
        "dropped_features": {
            "constant": dropped_constant,
            "high_corr": dropped_corr,
            "high_vif": dropped_vif,
        },
        "target_config": {
            "horizon": target_config.horizon,
            "threshold": target_config.threshold,
        },
    }
    save_json(metadata, reports_dir / "data_prep_metadata.json")
    return metadata

In [27]:
data_cfg = DataConfig()
target_cfg = TargetConfig(horizon=6, threshold=0.0075)
split_cfg = SplitConfig(train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)

metadata = prepare_datasets(
    data_config=data_cfg,
    target_config=target_cfg,
    split_config=split_cfg,
    corr_threshold=0.95,
    apply_vif=True,
    vif_threshold=10.0,
)
metadata


NameError: name 'load_ohlcv_csv' is not defined

In [28]:
cfg = DataConfig()
parquet_paths = [cfg.features_parquet, cfg.train_parquet, cfg.val_parquet, cfg.test_parquet]
for p in parquet_paths:
    print(p, "exists=", p.exists(), "size=", p.stat().st_size if p.exists() else None)

train_df = pd.read_parquet(cfg.train_parquet)
val_df = pd.read_parquet(cfg.val_parquet)
test_df = pd.read_parquet(cfg.test_parquet)
print("train/val/test rows:", len(train_df), len(val_df), len(test_df))


data\processed\features.parquet exists= False size= None
data\processed\train.parquet exists= False size= None
data\processed\val.parquet exists= False size= None
data\processed\test.parquet exists= False size= None


FileNotFoundError: [Errno 2] No such file or directory: 'data\\processed\\train.parquet'